# Train AnoDDPM on BMAD Brain FLAIR

Google Colab notebook for the explicit **AnoDDPM-style BMAD adaptation** in this repository. This is not ANDi and is not a clinical system. The notebook pins both the Git branch and the private Hugging Face dataset revision, keeps test data sealed, and selects checkpoints using validation only.

Start with `RUN_MODE = "smoke"`. Switch to `pilot` only after the smoke run completes.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/tanh1c/DACN.git"
BRANCH = "feat/anoddpm-notebook-training"
WORK_ROOT = Path("/content") if Path("/content").is_dir() else Path.cwd()
REPO_DIR = WORK_ROOT / "DACN"

if REPO_DIR.exists():
    status = subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if status:
        raise RuntimeError("Existing Colab checkout has local changes; restart the runtime.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "switch", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
os.chdir(REPO_DIR)
CODE_REVISION = subprocess.run(
    ["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True
).stdout.strip()
print(f"Using repository: {REPO_DIR} at {CODE_REVISION}")

In [ ]:
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        ".[train,hub]",
        "matplotlib>=3.8",
        "pandas>=2.0",
    ]
)
print("Dependencies installed.")

In [ ]:
import json
import platform
from datetime import datetime, timezone

import torch
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not enabled. In Colab choose Runtime > Change runtime type > T4 GPU."
    )
DEVICE = torch.device("cuda")
print(
    {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "gpu": torch.cuda.get_device_name(0),
    }
)

## Hugging Face access

In Colab, open the key icon (**Secrets**), add a secret named `HF_TOKEN`, and enable notebook access. Use a token that can read the private dataset. Never paste the token into a notebook cell or commit it.

In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
except ImportError:
    HF_TOKEN = getpass("Hugging Face token (hidden): ")
else:
    try:
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception as exc:
        raise RuntimeError("Add HF_TOKEN in Colab Secrets and enable notebook access.") from exc

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing.")
print("Hugging Face token loaded without displaying it.")

In [ ]:
from huggingface_hub import snapshot_download

from dacn.data.hub import validate_hf_export

DATASET_REPO = "dacn-bmad-team/bmad-flair-patient-disjoint-v1"
DATASET_REVISION = "350c74f2313e4e73ca65e7ffc6e9dcf74040b3b7"
DATASET_DIR = REPO_DIR / "data/shared/bmad-flair-patient-disjoint-v1"

snapshot_download(
    repo_id=DATASET_REPO,
    repo_type="dataset",
    revision=DATASET_REVISION,
    token=HF_TOKEN,
    local_dir=DATASET_DIR,
)
metadata = validate_hf_export(DATASET_DIR, verify_checksums=True, allow_hf_cache=True)
display(
    {
        split: {key: values[key] for key in ("samples", "normal", "anomaly")}
        for split, values in metadata["splits"].items()
    }
)

## Choose a run mode

- `smoke`: 128 normal training slices and 16 stratified validation slices, one epoch. It verifies the pipeline; its metrics are **not research results**.
- `pilot`: all 7,500 normal training slices and all 83 validation slices, three epochs. It can run for hours and may exceed a free Colab session.

In [ ]:
RUN_MODE = "smoke"  # Change to "pilot" only after smoke succeeds.

COMMON_CONFIG = {
    "seed": 17,
    "batch_size": 1,
    "validation_batch_size": 1,
    "learning_rate": 1e-4,
    "base_channels": 32,
    "noise_type": "simplex",
    "beta_schedule": "linear",
    "loss_type": "l2",
}
if RUN_MODE == "smoke":
    CONFIG = {
        **COMMON_CONFIG,
        "epochs": 1,
        "maximum_training_timestep": 100,
        "validation_reconstruction_steps": 25,
        "max_train_samples": 128,
        "max_validation_samples": 16,
        "log_every_steps": 16,
    }
elif RUN_MODE == "pilot":
    CONFIG = {
        **COMMON_CONFIG,
        "epochs": 3,
        "maximum_training_timestep": 800,
        "validation_reconstruction_steps": 100,
        "max_train_samples": None,
        "max_validation_samples": None,
        "log_every_steps": 100,
    }
else:
    raise ValueError("RUN_MODE must be 'smoke' or 'pilot'.")
display(CONFIG)

In [ ]:
USE_GOOGLE_DRIVE = True
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        OUTPUT_ROOT = Path("/content/drive/MyDrive/DACN/runs")
    except ImportError:
        OUTPUT_ROOT = REPO_DIR / "runs/colab"
else:
    OUTPUT_ROOT = REPO_DIR / "runs/colab"

RUN_ID = f"anoddpm-{RUN_MODE}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
if OUTPUT_DIR.exists():
    raise RuntimeError(f"Output directory already exists: {OUTPUT_DIR}")
print(f"Results will be written to: {OUTPUT_DIR}")

In [ ]:
from dacn.training.anoddpm import train

RESULT = train(
    train_dir=DATASET_DIR / "data/train",
    validation_dir=DATASET_DIR / "data/val",
    output_dir=OUTPUT_DIR,
    config=CONFIG,
    device=DEVICE,
)
display(RESULT)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history = json.loads((OUTPUT_DIR / "training_history.json").read_text())["epochs"]
rows = [
    {
        "epoch": item["epoch"],
        "train_loss": item["train_loss"],
        "pixel_ap": item["validation"]["pixel_ap"],
        "pixel_auroc": item["validation"]["pixel_auroc"],
        "image_ap": item["validation"]["image_ap"],
        "image_auroc": item["validation"]["image_auroc"],
        "seconds": item["seconds"],
    }
    for item in history
]
table = pd.DataFrame(rows)
table.to_csv(OUTPUT_DIR / "metrics_by_epoch.csv", index=False)
display(table.round(6))

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(table["epoch"], table["train_loss"], marker="o")
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Loss")
for column in ("pixel_ap", "pixel_auroc", "image_ap", "image_auroc"):
    axes[1].plot(table["epoch"], table[column], marker="o", label=column)
axes[1].set(title="Validation metrics", xlabel="Epoch", ylabel="Score", ylim=(0, 1))
axes[1].legend()
figure.tight_layout()
figure.savefig(OUTPUT_DIR / "training_curves.png", dpi=160, bbox_inches="tight")
plt.show()

manifest = {
    "schema_version": 1,
    "method": "anoddpm_bmad_flair_adaptation",
    "run_mode": RUN_MODE,
    "code_revision": CODE_REVISION,
    "dataset_repo": DATASET_REPO,
    "dataset_revision": DATASET_REVISION,
    "config": CONFIG,
    "result": RESULT,
    "gpu": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "completed_at": datetime.now(timezone.utc).isoformat(),
}
(OUTPUT_DIR / "notebook_run_manifest.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print(f"Saved checkpoint, table, chart, and manifest under {OUTPUT_DIR}")

In [ ]:
DOWNLOAD_RESULTS_ZIP = False
if DOWNLOAD_RESULTS_ZIP:
    import shutil

    archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
    try:
        from google.colab import files

        files.download(archive)
    except ImportError:
        print(f"Archive created at {archive}")
else:
    print(f"Results remain at {OUTPUT_DIR}. Set DOWNLOAD_RESULTS_ZIP=True to download them.")